# LeetCode 2032 — Two Out of Three bằng Python

Link đề: https://leetcode.com/problems/two-out-of-three/ · Độ khó: **Easy** · Chủ đề: Array, Hash Table, Bit Manipulation

Chữ ký hàm LeetCode (Python 3):

```python
class Solution:
    def twoOutOfThree(self, nums1: List[int], nums2: List[int], nums3: List[int]) -> List[int]:

```

**Mục tiêu học của notebook này**

1. Hiểu chính xác cụm "có mặt trong ít nhất 2 trên 3 mảng" — và vì sao phần tử **lặp lại trong cùng một mảng** không được tính hai lần.
2. Nắm chắc `set` (tập hợp): tạo, thêm, kiểm tra `in`, phép giao `&`, hợp `|`.
3. Biết dùng `dict` / `Counter` để **đếm số mảng** chứa mỗi giá trị.
4. Đi từ cách trực giác (duyệt mọi giá trị 1..100) lên cách đếm bằng set, cách phép toán tập hợp và cách **bitmask**.
5. Viết được `class Solution` nộp LeetCode, tự test bằng `assert` và tự làm các biến thể "k trên m mảng".

## 1. Đề bài yêu cầu gì?

Cho 3 mảng số nguyên `nums1`, `nums2`, `nums3`. Trả về **một mảng các giá trị phân biệt** gồm mọi giá trị xuất hiện trong **ít nhất 2 trong 3 mảng**. Thứ tự kết quả tùy ý.

| Ví dụ | nums1 | nums2 | nums3 | Output | Giải thích |
|---|---|---|---|---|---|
| 1 | `[1,1,3,2]` | `[2,3]` | `[3]` | `[3,2]` | 3 có ở cả 3 mảng; 2 có ở nums1 và nums2 |
| 2 | `[3,1]` | `[2,3]` | `[1,2]` | `[2,3,1]` | mỗi số có ở đúng 2 mảng |
| 3 | `[1,2,2]` | `[4,3,3]` | `[5]` | `[]` | 2 lặp **trong nums1** nhưng chỉ ở 1 mảng → không tính |

Constraints:

- `1 <= nums1.length, nums2.length, nums3.length <= 100`
- `1 <= nums1[i], nums2[j], nums3[k] <= 100`

Câu hỏi then chốt: **"có mặt trong một mảng"** nghĩa là xuất hiện ít nhất một lần trong mảng đó. Xuất hiện 2 lần trong cùng `nums1` vẫn chỉ là "1 mảng".

In [1]:
# Chạy thử ý nghĩa đề bằng tay trên ví dụ 3: số 2 xuất hiện mấy lần, ở mấy mảng?
nums1, nums2, nums3 = [1, 2, 2], [4, 3, 3], [5]
print("Số lần 2 xuất hiện trong nums1:", nums1.count(2))
print("2 có trong nums1?", 2 in nums1)
print("2 có trong nums2?", 2 in nums2)
print("2 có trong nums3?", 2 in nums3)
so_mang = (2 in nums1) + (2 in nums2) + (2 in nums3)
print("Số MẢNG chứa 2:", so_mang, "→ cần >= 2 nên KHÔNG lấy")

Số lần 2 xuất hiện trong nums1: 2
2 có trong nums1? True
2 có trong nums2? False
2 có trong nums3? False
Số MẢNG chứa 2: 1 → cần >= 2 nên KHÔNG lấy


Chú ý dòng `(2 in nums1) + (2 in nums2) + (2 in nums3)`: trong Python, `True` cộng như số `1`, `False` như `0`. Đây là mẹo nhỏ để **đếm có bao nhiêu điều kiện đúng**. Ta sẽ dùng lại nó ở cách 1.

## 2. Nền tảng Python cần biết

### 2.1. `list`, toán tử `in` và `True + True`

- `x in lst` trả về `True` nếu `x` nằm trong list. Python phải dò từ đầu đến cuối → tốn `O(len(lst))`.
- `bool` là "con" của `int`: `True == 1`, `False == 0`, nên cộng được.

In [2]:
lst = [4, 3, 3]
print(3 in lst, 9 in lst)        # True False
print(True + True + False)       # 2
print(int(True), int(False))     # 1 0
print(sum([True, False, True]))  # 2  -> sum cũng đếm được số True

True False
2
1 0
2


### 2.2. `set` — tập hợp, công cụ chính của bài

`set` là tập hợp **không có phần tử trùng** và **không giữ thứ tự**. Ba thao tác ta cần:

| Thao tác | Cú pháp | Ý nghĩa | Chi phí trung bình |
|---|---|---|---|
| Tạo từ list | `set([1,1,3,2])` | bỏ trùng → `{1,2,3}` | `O(n)` |
| Kiểm tra | `x in s` | có `x` không? | `O(1)` trung bình |
| Thêm | `s.add(x)` | thêm `x` (nếu đã có thì không đổi) | `O(1)` trung bình |

Đọc `{1, 2, 3}` là "tập gồm 1, 2, 3". Lưu ý `{}` là **dict rỗng**, còn tập rỗng phải viết `set()`.

In [3]:
s = set([1, 1, 3, 2])
print(s)                 # {1, 2, 3}  (thứ tự in có thể khác, không quan trọng)
print(len(s))            # 3: phần tử trùng đã bị gộp
s.add(3)                 # 3 đã có → không đổi
s.add(7)
print(sorted(s))         # [1, 2, 3, 7]  sorted() trả về list đã sắp xếp
print(type({}), type(set()))

{1, 2, 3}
3
[1, 2, 3, 7]
<class 'dict'> <class 'set'>


### 2.3. Phép toán tập hợp: giao `&`, hợp `|`

- `A & B` (giao, intersection): phần tử có trong **cả** A và B.
- `A | B` (hợp, union): phần tử có trong **ít nhất một** trong A, B.
- `A - B` (hiệu): có trong A nhưng không có trong B.

Với bài này: "có trong ít nhất 2 mảng" = có trong (1 và 2) **hoặc** (1 và 3) **hoặc** (2 và 3) = `(A & B) | (A & C) | (B & C)`.

In [4]:
A, B_, C_ = {1, 2, 3}, {2, 3}, {3}
print("A & B =", A & B_)                 # {2, 3}
print("A & C =", A & C_)                 # {3}
print("B & C =", B_ & C_)                # {3}
print("hợp 3 phần giao =", (A & B_) | (A & C_) | (B_ & C_))   # {2, 3}
print("A - B =", A - B_)                 # {1}

A & B = {2, 3}
A & C = {3}
B & C = {3}
hợp 3 phần giao = {2, 3}
A - B = {1}


### 2.4. `dict.get` và `collections.Counter` để đếm

Một `dict` lưu cặp `key: value`. Để đếm, ta hay viết `cnt[x] = cnt.get(x, 0) + 1`:
- `cnt.get(x, 0)` đọc giá trị của `x`, nếu chưa có thì trả `0` (không báo lỗi như `cnt[x]`).

`Counter` (trong module chuẩn `collections`) làm sẵn việc đó: `Counter(iterable)` đếm số lần mỗi phần tử.

In [5]:
from collections import Counter

cnt = {}
for x in [1, 2, 2, 3]:
    cnt[x] = cnt.get(x, 0) + 1
print(cnt)                         # {1: 1, 2: 2, 3: 1}

print(Counter([1, 2, 2, 3]))       # Counter({2: 2, 1: 1, 3: 1})
# Ghép 3 set lại thành 1 list rồi đếm: mỗi giá trị được đếm tối đa 1 lần / mảng
print(Counter(list({1, 3, 2}) + list({2, 3}) + list({3})))

{1: 1, 2: 2, 3: 1}
Counter({2: 2, 1: 1, 3: 1})
Counter({3: 3, 2: 2, 1: 1})


### 2.5. List comprehension

`[x for x in cnt if cnt[x] >= 2]` đọc là: "lấy mọi `x` trong `cnt` **sao cho** `cnt[x] >= 2`". Nó tương đương vòng `for` + `if` + `append` nhưng ngắn hơn.

In [6]:
cnt = {1: 1, 2: 2, 3: 3}
ket_qua = []
for x in cnt:
    if cnt[x] >= 2:
        ket_qua.append(x)
print(ket_qua)
print([x for x in cnt if cnt[x] >= 2])   # cùng kết quả

[2, 3]
[2, 3]


### 2.6. Bit và bitmask (dùng ở cách 4)

Mỗi số nguyên có biểu diễn nhị phân. Ta dùng 3 bit để ghi "giá trị này đã có ở mảng nào":

| bit | giá trị | nghĩa |
|---|---|---|
| bit 0 | `1 << 0 = 1` (`001`) | có trong nums1 |
| bit 1 | `1 << 1 = 2` (`010`) | có trong nums2 |
| bit 2 | `1 << 2 = 4` (`100`) | có trong nums3 |

- `a << i`: dịch trái `i` bit, tức nhân 2^i.
- `mask |= bit`: bật bit (OR). Bật lại bit đã bật thì không đổi → tự động "bỏ trùng".
- `bin(x).count("1")` hoặc `x.bit_count()` (Python 3.10+): đếm số bit 1.

In [7]:
mask = 0
mask |= 1 << 0      # có ở nums1
mask |= 1 << 0      # lại ở nums1 → không đổi
mask |= 1 << 2      # có ở nums3
print(mask, bin(mask))          # 5 0b101
print(bin(mask).count("1"))     # 2 mảng

5 0b101
2


## 3. Cách 1 — Trực giác: thử mọi giá trị có thể (1..100)

Constraint nói mọi phần tử nằm trong `[1, 100]`. Vậy ta có thể hỏi từng giá trị `v = 1, 2, ..., 100`:

> "`v` có trong nums1 không? trong nums2 không? trong nums3 không?" → đếm số câu "có". Nếu `>= 2` thì lấy `v`.

Cách này **tự động đúng với phần tử trùng**, vì `v in nums1` chỉ trả lời có/không, không quan tâm `v` xuất hiện mấy lần.

In [8]:
def two_out_of_three_bruteforce(nums1, nums2, nums3):
    result = []
    for v in range(1, 101):                     # mọi giá trị hợp lệ theo constraints
        so_mang = (v in nums1) + (v in nums2) + (v in nums3)
        if so_mang >= 2:
            result.append(v)
    return result

print(two_out_of_three_bruteforce([1, 1, 3, 2], [2, 3], [3]))       # [2, 3]
print(two_out_of_three_bruteforce([3, 1], [2, 3], [1, 2]))          # [1, 2, 3]
print(two_out_of_three_bruteforce([1, 2, 2], [4, 3, 3], [5]))       # []

[2, 3]
[1, 2, 3]
[]


### Chi phí của cách 1

Gọi `V = 100` là số giá trị có thể, `n = len(nums1)+len(nums2)+len(nums3)`. Mỗi `v in nums` tốn tới `O(độ dài mảng)` → tổng `O(V · n)` = khoảng 100 × 300 = 30.000 phép so sánh. Với constraints nhỏ thì vẫn qua, nhưng:

- Nó phụ thuộc vào việc **biết trước miền giá trị** `1..100`. Nếu giá trị tới `10^9` thì không thể duyệt hết.
- Ta hỏi `in` trên **list** nhiều lần — mỗi lần dò lại từ đầu.

## 4. Nút thắt và insight

**Nút thắt 1:** `v in list` chậm (O(độ dài)). Đổi mỗi mảng thành `set` thì `in` còn `O(1)` trung bình.

**Nút thắt 2:** duyệt "mọi giá trị có thể" là thừa. Chỉ cần duyệt **các giá trị thực sự xuất hiện**.

**Insight quan trọng nhất:** nếu ta đếm "số lần xuất hiện" trên cả 3 mảng gộp lại, ví dụ `[1,2,2] + [4,3,3] + [5]`, thì số 2 được đếm **2 lần** dù chỉ ở 1 mảng → sai. Cách sửa: **bỏ trùng trong từng mảng trước** (`set(nums1)`), rồi mới đếm. Khi đó mỗi giá trị đóng góp tối đa 1 cho mỗi mảng, và "số lần đếm được" = "số mảng chứa nó".

In [9]:
from collections import Counter
nums1, nums2, nums3 = [1, 2, 2], [4, 3, 3], [5]

sai = Counter(nums1 + nums2 + nums3)
print("Đếm KHÔNG bỏ trùng:", dict(sai))            # 2 -> 2 lần, 3 -> 2 lần (sai!)
dung = Counter(list(set(nums1)) + list(set(nums2)) + list(set(nums3)))
print("Đếm SAU khi bỏ trùng:", dict(dung))        # mọi số đều 1

Đếm KHÔNG bỏ trùng: {1: 1, 2: 2, 4: 1, 3: 2, 5: 1}
Đếm SAU khi bỏ trùng: {1: 1, 2: 1, 3: 1, 4: 1, 5: 1}


## 5. Cách 2 (lời giải chính) — set cho từng mảng + đếm số mảng

Ghép từng mảnh:

1. `for arr in (nums1, nums2, nums3)`: duyệt lần lượt 3 mảng.
2. `for v in set(arr)`: duyệt các giá trị **phân biệt** của mảng đó.
3. `cnt[v] = cnt.get(v, 0) + 1`: tăng "số mảng chứa v".
4. Cuối cùng lấy mọi `v` có `cnt[v] >= 2`.

Mảnh 1 + 2 nhìn riêng:

In [10]:
for idx, arr in enumerate(([1, 1, 3, 2], [2, 3], [3]), start=1):
    print(f"nums{idx} = {arr} -> set = {set(arr)}")

nums1 = [1, 1, 3, 2] -> set = {1, 2, 3}
nums2 = [2, 3] -> set = {2, 3}
nums3 = [3] -> set = {3}


`enumerate(seq, start=1)` trả về từng cặp `(chỉ số, phần tử)` với chỉ số bắt đầu từ 1 — tiện để in tên `nums1, nums2, nums3`.

Giờ ghép thành hàm hoàn chỉnh:

In [11]:
def two_out_of_three_count(nums1, nums2, nums3):
    cnt = {}                                  # giá trị -> số mảng chứa nó
    for arr in (nums1, nums2, nums3):
        for v in set(arr):                    # bỏ trùng trong CÙNG một mảng
            cnt[v] = cnt.get(v, 0) + 1
    return [v for v in cnt if cnt[v] >= 2]

print(sorted(two_out_of_three_count([1, 1, 3, 2], [2, 3], [3])))     # [2, 3]
print(sorted(two_out_of_three_count([3, 1], [2, 3], [1, 2])))        # [1, 2, 3]
print(sorted(two_out_of_three_count([1, 2, 2], [4, 3, 3], [5])))     # []

[2, 3]
[1, 2, 3]
[]


Ta in `sorted(...)` chỉ để dễ so sánh bằng mắt; đề cho phép thứ tự tùy ý nên LeetCode tự chấp nhận mọi thứ tự.

## 6. TRACE từng bước — nhìn `cnt` thay đổi

Ví dụ 1: `nums1 = [1,1,3,2]`, `nums2 = [2,3]`, `nums3 = [3]`.

| Bước | Mảng | set(mảng) | v | cnt trước | cnt sau |
|---|---|---|---|---|---|
| 1 | nums1 | {1,2,3} | 1 | {} | {1:1} |
| 2 | nums1 | {1,2,3} | 2 | {1:1} | {1:1, 2:1} |
| 3 | nums1 | {1,2,3} | 3 | {1:1, 2:1} | {1:1, 2:1, 3:1} |
| 4 | nums2 | {2,3} | 2 | ... | {1:1, 2:2, 3:1} |
| 5 | nums2 | {2,3} | 3 | ... | {1:1, 2:2, 3:2} |
| 6 | nums3 | {3} | 3 | ... | {1:1, 2:2, 3:3} |

Lọc `cnt[v] >= 2` → `{2, 3}`. Để ý: số `1` xuất hiện **2 lần** trong nums1 nhưng `cnt[1]` chỉ là 1 — đó chính là tác dụng của `set`.

In [12]:
def trace_two_out_of_three(nums1, nums2, nums3):
    cnt = {}
    step = 1
    for name, arr in (("nums1", nums1), ("nums2", nums2), ("nums3", nums3)):
        uniq = sorted(set(arr))
        print(f"--- {name} = {arr} -> các giá trị phân biệt {uniq}")
        for v in uniq:
            truoc = dict(cnt)
            cnt[v] = cnt.get(v, 0) + 1
            print(f"Bước {step}: v={v}, cnt trước={truoc} -> cnt sau={cnt}")
            step += 1
    ans = [v for v in cnt if cnt[v] >= 2]
    print("Giữ các v có cnt >= 2:", sorted(ans))
    return ans

trace_two_out_of_three([1, 1, 3, 2], [2, 3], [3])

--- nums1 = [1, 1, 3, 2] -> các giá trị phân biệt [1, 2, 3]
Bước 1: v=1, cnt trước={} -> cnt sau={1: 1}
Bước 2: v=2, cnt trước={1: 1} -> cnt sau={1: 1, 2: 1}
Bước 3: v=3, cnt trước={1: 1, 2: 1} -> cnt sau={1: 1, 2: 1, 3: 1}
--- nums2 = [2, 3] -> các giá trị phân biệt [2, 3]
Bước 4: v=2, cnt trước={1: 1, 2: 1, 3: 1} -> cnt sau={1: 1, 2: 2, 3: 1}
Bước 5: v=3, cnt trước={1: 1, 2: 2, 3: 1} -> cnt sau={1: 1, 2: 2, 3: 2}
--- nums3 = [3] -> các giá trị phân biệt [3]
Bước 6: v=3, cnt trước={1: 1, 2: 2, 3: 2} -> cnt sau={1: 1, 2: 2, 3: 3}
Giữ các v có cnt >= 2: [2, 3]


[2, 3]

Trace thêm ví dụ 3 — nơi người mới dễ sai vì số 2 và 3 bị lặp trong cùng mảng:

In [13]:
trace_two_out_of_three([1, 2, 2], [4, 3, 3], [5])

--- nums1 = [1, 2, 2] -> các giá trị phân biệt [1, 2]
Bước 1: v=1, cnt trước={} -> cnt sau={1: 1}
Bước 2: v=2, cnt trước={1: 1} -> cnt sau={1: 1, 2: 1}
--- nums2 = [4, 3, 3] -> các giá trị phân biệt [3, 4]
Bước 3: v=3, cnt trước={1: 1, 2: 1} -> cnt sau={1: 1, 2: 1, 3: 1}
Bước 4: v=4, cnt trước={1: 1, 2: 1, 3: 1} -> cnt sau={1: 1, 2: 1, 3: 1, 4: 1}
--- nums3 = [5] -> các giá trị phân biệt [5]
Bước 5: v=5, cnt trước={1: 1, 2: 1, 3: 1, 4: 1} -> cnt sau={1: 1, 2: 1, 3: 1, 4: 1, 5: 1}
Giữ các v có cnt >= 2: []


[]

## 7. Cách 3 — Phép toán tập hợp (gọn nhất)

Đặt `A = set(nums1)`, `B = set(nums2)`, `C = set(nums3)`. "Có trong ít nhất 2 tập" nghĩa là thuộc ít nhất một trong ba cặp giao:

```
(A & B) | (A & C) | (B & C)
```

Lý do: nếu `v` thuộc ≥ 2 tập thì có một cặp tập cùng chứa `v` → `v` nằm trong giao của cặp đó. Ngược lại, nếu `v` nằm trong một giao cặp thì hiển nhiên thuộc ≥ 2 tập.

In [14]:
def two_out_of_three_setops(nums1, nums2, nums3):
    A, B, C = set(nums1), set(nums2), set(nums3)
    return list((A & B) | (A & C) | (B & C))

print(sorted(two_out_of_three_setops([1, 1, 3, 2], [2, 3], [3])))
print(sorted(two_out_of_three_setops([3, 1], [2, 3], [1, 2])))
print(sorted(two_out_of_three_setops([1, 2, 2], [4, 3, 3], [5])))

[2, 3]
[1, 2, 3]
[]


Cách viết tương đương khác bạn có thể gặp: `(A & B) | (C & (A | B))` — "có ở A và B, **hoặc** có ở C và ở một trong A/B".

In [15]:
A, B_, C_ = {1, 2, 3}, {2, 3}, {3}
print((A & B_) | (C_ & (A | B_)))   # {2, 3} - cùng kết quả

{2, 3}


## 8. Cách 4 — Bitmask: mỗi giá trị nhớ "đã thấy ở mảng nào"

Thay vì đếm, ta lưu cho mỗi giá trị `v` một số `mask[v]` 3 bit. Mảng thứ `i` (0, 1, 2) bật bit `i`. Vì OR một bit đã bật không đổi gì, phần tử trùng trong cùng mảng **tự động không bị đếm thêm** — không cần `set`!

Cuối cùng giữ `v` có ít nhất 2 bit bật. Mẹo kiểm tra "≥ 2 bit" không cần đếm: `m & (m - 1) != 0` (xóa bit thấp nhất mà vẫn còn bit → có ≥ 2 bit).

In [16]:
for m in [0b001, 0b010, 0b011, 0b101, 0b111, 0b100]:
    print(f"m={m:03b}  m&(m-1)={m & (m - 1):03b}  >=2 bit? {m & (m - 1) != 0}")

m=001  m&(m-1)=000  >=2 bit? False
m=010  m&(m-1)=000  >=2 bit? False
m=011  m&(m-1)=010  >=2 bit? True
m=101  m&(m-1)=100  >=2 bit? True
m=111  m&(m-1)=110  >=2 bit? True
m=100  m&(m-1)=000  >=2 bit? False


In [17]:
def two_out_of_three_bitmask(nums1, nums2, nums3):
    mask = [0] * 101                      # chỉ số 0..100, giá trị nằm trong 1..100
    for i, arr in enumerate((nums1, nums2, nums3)):
        for v in arr:
            mask[v] |= 1 << i             # bật bit i: "v có trong mảng i"
    return [v for v in range(1, 101) if mask[v] & (mask[v] - 1)]

print(two_out_of_three_bitmask([1, 1, 3, 2], [2, 3], [3]))
print(two_out_of_three_bitmask([3, 1], [2, 3], [1, 2]))
print(two_out_of_three_bitmask([1, 2, 2], [4, 3, 3], [5]))

[2, 3]
[1, 2, 3]
[]


Trace bitmask cho ví dụ 1 (chỉ in các giá trị có xuất hiện):

In [18]:
def trace_bitmask(nums1, nums2, nums3):
    mask = [0] * 101
    for i, arr in enumerate((nums1, nums2, nums3)):
        for v in arr:
            old = mask[v]
            mask[v] |= 1 << i
            print(f"mảng {i}, v={v}: mask {old:03b} -> {mask[v]:03b}")
    for v in range(1, 101):
        if mask[v]:
            print(f"v={v}: mask={mask[v]:03b}, số mảng={bin(mask[v]).count('1')}")

trace_bitmask([1, 1, 3, 2], [2, 3], [3])

mảng 0, v=1: mask 000 -> 001
mảng 0, v=1: mask 001 -> 001
mảng 0, v=3: mask 000 -> 001
mảng 0, v=2: mask 000 -> 001
mảng 1, v=2: mask 001 -> 011
mảng 1, v=3: mask 001 -> 011
mảng 2, v=3: mask 011 -> 111
v=1: mask=001, số mảng=1
v=2: mask=011, số mảng=2
v=3: mask=111, số mảng=3


Nhìn dòng `mảng 0, v=1` lần thứ hai: mask vẫn `001 -> 001`. Đó là "bỏ trùng miễn phí".

## 9. Code nộp LeetCode

Bản chính (cách 2 — set + đếm). LeetCode đã tự import `List`; khi chạy ngoài LeetCode ta tự `from typing import List`.

In [19]:
from typing import List

class Solution:
    def twoOutOfThree(self, nums1: List[int], nums2: List[int], nums3: List[int]) -> List[int]:
        cnt = {}
        for arr in (nums1, nums2, nums3):
            for v in set(arr):
                cnt[v] = cnt.get(v, 0) + 1
        return [v for v in cnt if cnt[v] >= 2]

sol = Solution()
print(sorted(sol.twoOutOfThree([1, 1, 3, 2], [2, 3], [3])))

[2, 3]


Bản thay thế — phép toán tập hợp (cách 3), cũng nộp được:

In [20]:
class SolutionSetOps:
    def twoOutOfThree(self, nums1: List[int], nums2: List[int], nums3: List[int]) -> List[int]:
        a, b, c = set(nums1), set(nums2), set(nums3)
        return list((a & b) | (a & c) | (b & c))

print(sorted(SolutionSetOps().twoOutOfThree([3, 1], [2, 3], [1, 2])))

[1, 2, 3]


## 10. Vì sao lời giải đúng?

**Bất biến (invariant)** của cách 2: sau khi xử lý xong `t` mảng đầu tiên, `cnt[v]` = **số mảng trong `t` mảng đó có chứa `v`**.

- Ban đầu `t = 0`: `cnt` rỗng, mọi giá trị có 0 mảng → đúng.
- Xử lý mảng thứ `t+1`: ta duyệt `set(arr)`, nên mỗi `v` có trong mảng được `+1` **đúng một lần**, `v` không có thì không đổi → bất biến giữ cho `t+1`.
- Sau 3 mảng: `cnt[v]` = số mảng chứa `v`. Lấy `cnt[v] >= 2` chính là định nghĩa đề.

**Không trả trùng:** keys của `dict` là duy nhất, nên mỗi `v` xuất hiện trong kết quả tối đa 1 lần — đúng yêu cầu "distinct array".

**Không bỏ sót:** mọi `v` xuất hiện ở ≥ 1 mảng đều là key của `cnt`, nên đều được xét.

## 11. Độ phức tạp và so sánh các cách

Gọi `n = len(nums1) + len(nums2) + len(nums3)` (≤ 300), `V = 100` là kích thước miền giá trị.

| Cách | Ý tưởng | Time | Space | Ghi chú |
|---|---|---|---|---|
| 1. Thử 1..100 | `(v in nums1)+...` | `O(V·n)` | `O(1)` ngoài output | phụ thuộc miền giá trị nhỏ |
| 2. set + đếm | `cnt[v]` = số mảng | `O(n)` trung bình | `O(n)` | tổng quát, dễ mở rộng "k trên m" |
| 3. Phép toán tập | `(A&B)|(A&C)|(B&C)` | `O(n)` trung bình | `O(n)` | ngắn nhất cho đúng 3 mảng |
| 4. Bitmask | `mask[v] |= 1<<i` | `O(n + V)` | `O(V)` | không cần set, biết mảng nào chứa v |

"Trung bình" vì thao tác `set`/`dict` là O(1) trung bình (băm/hash), không phải tuyệt đối trong mọi trường hợp.

## 12. Bộ test

Nhóm test: (a) 3 ví dụ đề, (b) biên — mảng 1 phần tử, mọi mảng giống nhau, (c) phản ví dụ "trùng trong cùng mảng", (d) đối chiếu 4 cách trên dữ liệu ngẫu nhiên có seed. Vì thứ tự tùy ý, ta so sánh `sorted(...)`.

In [21]:
def check(name, actual, expected):
    assert sorted(actual) == sorted(expected), (name, actual, expected)
    print(f"{name:<38} PASS  -> {sorted(actual)}")

sol = Solution()
cases = [
    ("ví dụ 1",                    [1, 1, 3, 2], [2, 3], [3], [2, 3]),
    ("ví dụ 2",                    [3, 1], [2, 3], [1, 2], [1, 2, 3]),
    ("ví dụ 3 (trùng trong mảng)", [1, 2, 2], [4, 3, 3], [5], []),
    ("mỗi mảng 1 phần tử, giống",  [7], [7], [7], [7]),
    ("mỗi mảng 1 phần tử, khác",   [1], [2], [3], []),
    ("chỉ nums2 & nums3 chung",    [9], [5, 5, 5], [5], [5]),
    ("giá trị biên 1 và 100",      [1, 100], [100], [1], [1, 100]),
]
for name, a, b_, c_, exp in cases:
    check(name, sol.twoOutOfThree(a, b_, c_), exp)

ví dụ 1                                PASS  -> [2, 3]
ví dụ 2                                PASS  -> [1, 2, 3]
ví dụ 3 (trùng trong mảng)             PASS  -> []
mỗi mảng 1 phần tử, giống              PASS  -> [7]
mỗi mảng 1 phần tử, khác               PASS  -> []
chỉ nums2 & nums3 chung                PASS  -> [5]
giá trị biên 1 và 100                  PASS  -> [1, 100]


In [22]:
import random
random.seed(2032)
funcs = [two_out_of_three_bruteforce, two_out_of_three_count,
         two_out_of_three_setops, two_out_of_three_bitmask,
         Solution().twoOutOfThree, SolutionSetOps().twoOutOfThree]
for _ in range(500):
    arrs = [[random.randint(1, 15) for _ in range(random.randint(1, 8))] for _ in range(3)]
    answers = [sorted(f(*arrs)) for f in funcs]
    assert all(ans == answers[0] for ans in answers), (arrs, answers)
print("500 bộ ngẫu nhiên: cả 6 cài đặt cho cùng kết quả — PASS")

500 bộ ngẫu nhiên: cả 6 cài đặt cho cùng kết quả — PASS


In [23]:
# Tính chất: kết quả không có phần tử trùng, và mỗi phần tử thật sự nằm trong >= 2 mảng
random.seed(7)
for _ in range(200):
    arrs = [[random.randint(1, 10) for _ in range(random.randint(1, 6))] for _ in range(3)]
    res = Solution().twoOutOfThree(*arrs)
    assert len(res) == len(set(res))
    for v in res:
        assert sum(v in arr for arr in arrs) >= 2
print("Tính chất distinct + đúng định nghĩa — PASS")

Tính chất distinct + đúng định nghĩa — PASS


## 13. Lỗi người mới hay gặp

### Lỗi 1: Đếm số lần xuất hiện thay vì số mảng

```python
cnt = Counter(nums1 + nums2 + nums3)
return [v for v in cnt if cnt[v] >= 2]
```

Sai vì số lặp trong **cùng một mảng** bị đếm nhiều lần. Ví dụ 3 sẽ trả `[2, 3]` thay vì `[]`.

In [24]:
def wrong_count_all(nums1, nums2, nums3):
    cnt = Counter(nums1 + nums2 + nums3)
    return [v for v in cnt if cnt[v] >= 2]

print("Sai :", wrong_count_all([1, 2, 2], [4, 3, 3], [5]))
print("Đúng:", two_out_of_three_count([1, 2, 2], [4, 3, 3], [5]))

Sai : [2, 3]
Đúng: []


**Sửa:** bỏ trùng trước: `Counter(list(set(nums1)) + list(set(nums2)) + list(set(nums3)))`.

### Lỗi 2: Trả kết quả bị lặp

```python
res = []
for v in nums1 + nums2 + nums3:
    if (v in nums1) + (v in nums2) + (v in nums3) >= 2:
        res.append(v)
```

Điều kiện đúng nhưng `v` được thêm mỗi lần nó xuất hiện → kết quả có phần tử trùng, vi phạm "distinct".

In [25]:
def wrong_duplicates(nums1, nums2, nums3):
    res = []
    for v in nums1 + nums2 + nums3:
        if (v in nums1) + (v in nums2) + (v in nums3) >= 2:
            res.append(v)
    return res

print("Sai :", wrong_duplicates([1, 1, 3, 2], [2, 3], [3]))     # 3 bị lặp nhiều lần
print("Sửa :", sorted(set(wrong_duplicates([1, 1, 3, 2], [2, 3], [3]))))

Sai : [3, 2, 2, 3, 3]
Sửa : [2, 3]


### Lỗi 3: Chỉ lấy giao của cả 3 tập

`A & B & C` là "có ở **cả 3** mảng", không phải "ít nhất 2". Ví dụ 2 sẽ trả `[]` thay vì `[1,2,3]`.

### Lỗi 4: Viết `{}` để tạo tập rỗng

`s = {}` tạo **dict**, gọi `s.add(1)` sẽ lỗi `AttributeError`. Phải dùng `s = set()`.

In [26]:
A, B_, C_ = {3, 1}, {2, 3}, {1, 2}
print("Sai (giao cả 3):", A & B_ & C_)
print("Đúng:", (A & B_) | (A & C_) | (B_ & C_))
s = {}
print("type({}) =", type(s).__name__, "-> dùng set() cho tập rỗng:", type(set()).__name__)

Sai (giao cả 3): set()
Đúng: {1, 2, 3}
type({}) = dict -> dùng set() cho tập rỗng: set


## 14. Mẹo nhận dạng pattern

- Đề nói "**xuất hiện trong ít nhất k mảng/nhóm**" → nghĩ ngay: **bỏ trùng trong từng nhóm** (set) rồi **đếm số nhóm** (dict/Counter).
- Đề có "**distinct**", "**unique**", "**không trùng**" → `set` hoặc keys của `dict`.
- Số nhóm nhỏ (≤ 30) và cần biết **chính xác nhóm nào** chứa phần tử → **bitmask** mỗi nhóm 1 bit.
- Miền giá trị nhỏ (≤ 10^5) → có thể dùng **mảng đếm** `cnt = [0]*(MAX+1)` thay cho dict.
- Chỉ có 2–3 tập và câu hỏi là "chung" / "riêng" → phép toán `&`, `|`, `-`, `^`.

## 15. Biến thể

1. **k trên m mảng:** cho list các mảng `arrays` và số `k`, trả các giá trị có trong ít nhất `k` mảng. Cách 2 mở rộng ngay: đổi `>= 2` thành `>= k`.
2. **Có trong đúng 1 mảng** (riêng của một mảng): `cnt[v] == 1`.
3. **Có trong cả 3 mảng:** `A & B & C` (LeetCode 2248 "Intersection of Multiple Arrays" là phiên bản nhiều mảng, kết quả sắp xếp).
4. **Kèm thông tin mảng nào:** trả dict `v -> danh sách chỉ số mảng`, dùng bitmask rồi giải mã bit.

In [27]:
def at_least_k_of_m(arrays, k):
    cnt = {}
    for arr in arrays:
        for v in set(arr):
            cnt[v] = cnt.get(v, 0) + 1
    return sorted(v for v in cnt if cnt[v] >= k)

arrays = [[1, 2, 3], [2, 3, 4], [3, 4, 5], [3, 9]]
for k in range(1, 5):
    print(f"k={k}:", at_least_k_of_m(arrays, k))

k=1: [1, 2, 3, 4, 5, 9]
k=2: [2, 3, 4]
k=3: [3]
k=4: [3]


## 16. Bài tập tự luyện (tăng dần)

**Bài 1 — Đoán output (tính tay).** Không chạy code, hãy đoán: `set([5, 5, 1])`, `{1,2} & {2,3}`, `{1,2} | {2,3}`, `{1,2,3} - {2}`, `len(set("banana"))`.

**Bài 2 — Tính tay bảng `cnt`.** Với `nums1=[4,4,1]`, `nums2=[1,2]`, `nums3=[2,4,4]`, lập bảng như mục 6 và cho biết kết quả.

**Bài 3 — TODO:** hoàn thiện `exercise_two_out_of_three` bằng cách set + đếm.

**Bài 4 — TODO biến thể:** `only_one_array(nums1, nums2, nums3)` trả (sắp xếp tăng) các giá trị có trong **đúng 1** mảng.

**Bài 5 — TODO:** `which_arrays(nums1, nums2, nums3)` trả dict `v -> list chỉ số mảng (0,1,2)` cho **các v có trong ≥ 2 mảng**, dùng bitmask.

**Bài 6 — Mini challenge:** `at_least_k(arrays, k)` cho số mảng tùy ý, nhưng **không dùng set** — chỉ được dùng dict lưu "mảng cuối cùng đã đếm v" để tránh đếm trùng.

In [28]:
def exercise_two_out_of_three(nums1, nums2, nums3):
    # TODO 1: tạo dict cnt rỗng
    # TODO 2: duyệt từng mảng arr trong (nums1, nums2, nums3)
    # TODO 3: duyệt v trong set(arr), tăng cnt[v]
    # TODO 4: trả các v có cnt[v] >= 2
    pass

# Làm xong rồi mở test:
# assert sorted(exercise_two_out_of_three([1,1,3,2],[2,3],[3])) == [2, 3]
# assert exercise_two_out_of_three([1,2,2],[4,3,3],[5]) == []

In [29]:
def only_one_array(nums1, nums2, nums3):
    # TODO: giống bài 3 nhưng điều kiện là "đúng 1 mảng"; nhớ sắp xếp kết quả
    pass

# assert only_one_array([1,1,3,2],[2,3],[3]) == [1]
# assert only_one_array([1,2,2],[4,3,3],[5]) == [1, 2, 3, 4, 5]

In [30]:
def which_arrays(nums1, nums2, nums3):
    # TODO 1: mask[v] |= 1 << i cho mảng thứ i
    # TODO 2: với mask có >= 2 bit, giải mã các bit i đang bật
    pass

# assert which_arrays([1,1,3,2],[2,3],[3]) == {2: [0, 1], 3: [0, 1, 2]}

In [31]:
def at_least_k(arrays, k):
    # TODO: không dùng set. Gợi ý: last_seen[v] = chỉ số mảng gần nhất đã đếm v
    pass

# assert at_least_k([[1,1,2],[2,2,3],[3,1]], 2) == [1, 2, 3]
# assert at_least_k([[1,1,2],[2,2,3],[3,1]], 3) == []

## 17. Gợi ý 3 mức

| Bài | Mức 1 — hướng nghĩ | Mức 2 — cấu trúc | Mức 3 — gần code |
|---|---|---|---|
| 1 | set bỏ trùng, & là chung, \| là gộp | "banana" có những chữ cái nào? | `{5,1}`, `{2}`, `{1,2,3}`, `{1,3}`, `3` |
| 2 | bỏ trùng từng mảng trước | set: {1,4}, {1,2}, {2,4} | mỗi số 1, 2, 4 có ở 2 mảng |
| 3 | xem lại mục 5 | 2 vòng for lồng nhau + dict | `cnt[v] = cnt.get(v, 0) + 1` |
| 4 | chỉ đổi điều kiện lọc | `cnt[v] == 1` | `sorted(v for v in cnt if cnt[v] == 1)` |
| 5 | mask có ≥ 2 bit | `m & (m-1)` | `[i for i in range(3) if m >> i & 1]` |
| 6 | trùng trong cùng mảng = cùng chỉ số mảng | `if last_seen.get(v) != i:` | tăng cnt rồi gán `last_seen[v] = i` |

## 18. Đáp án tham khảo (đặt cuối để không lộ trước khi tự làm)

**Bài 1:** `{1, 5}` · `{2}` · `{1, 2, 3}` · `{1, 3}` · `3` (các chữ b, a, n).

**Bài 2:** set lần lượt `{1,4}`, `{1,2}`, `{2,4}` → `cnt = {1:2, 4:2, 2:2}` → kết quả `[1, 2, 4]` (thứ tự tùy ý).

In [32]:
print(set([5, 5, 1]), {1, 2} & {2, 3}, {1, 2} | {2, 3}, {1, 2, 3} - {2}, len(set("banana")))
print(sorted(two_out_of_three_count([4, 4, 1], [1, 2], [2, 4, 4])))

{1, 5} {2} {1, 2, 3} {1, 3} 3
[1, 2, 4]


In [33]:
def exercise_two_out_of_three_answer(nums1, nums2, nums3):
    cnt = {}
    for arr in (nums1, nums2, nums3):
        for v in set(arr):
            cnt[v] = cnt.get(v, 0) + 1
    return [v for v in cnt if cnt[v] >= 2]

def only_one_array_answer(nums1, nums2, nums3):
    cnt = {}
    for arr in (nums1, nums2, nums3):
        for v in set(arr):
            cnt[v] = cnt.get(v, 0) + 1
    return sorted(v for v in cnt if cnt[v] == 1)

def which_arrays_answer(nums1, nums2, nums3):
    mask = {}
    for i, arr in enumerate((nums1, nums2, nums3)):
        for v in arr:
            mask[v] = mask.get(v, 0) | (1 << i)
    return {v: [i for i in range(3) if m >> i & 1]
            for v, m in sorted(mask.items()) if m & (m - 1)}

def at_least_k_answer(arrays, k):
    cnt, last_seen = {}, {}
    for i, arr in enumerate(arrays):
        for v in arr:
            if last_seen.get(v) != i:        # lần đầu gặp v trong mảng i
                last_seen[v] = i
                cnt[v] = cnt.get(v, 0) + 1
    return sorted(v for v in cnt if cnt[v] >= k)

assert sorted(exercise_two_out_of_three_answer([1, 1, 3, 2], [2, 3], [3])) == [2, 3]
assert exercise_two_out_of_three_answer([1, 2, 2], [4, 3, 3], [5]) == []
assert only_one_array_answer([1, 1, 3, 2], [2, 3], [3]) == [1]
assert only_one_array_answer([1, 2, 2], [4, 3, 3], [5]) == [1, 2, 3, 4, 5]
assert which_arrays_answer([1, 1, 3, 2], [2, 3], [3]) == {2: [0, 1], 3: [0, 1, 2]}
assert at_least_k_answer([[1, 1, 2], [2, 2, 3], [3, 1]], 2) == [1, 2, 3]
assert at_least_k_answer([[1, 1, 2], [2, 2, 3], [3, 1]], 3) == []
random.seed(99)
for _ in range(200):
    arrs = [[random.randint(1, 8) for _ in range(random.randint(1, 5))] for _ in range(4)]
    for k in range(1, 5):
        assert at_least_k_answer(arrs, k) == at_least_k_of_m(arrs, k)
print("Tất cả đáp án bài tập — PASS")

Tất cả đáp án bài tập — PASS


## 19. Tổng kết — template nhớ nhanh + checklist submit

**Template "xuất hiện trong ít nhất k nhóm":**

```python
cnt = {}
for group in groups:
    for v in set(group):          # 1) bỏ trùng TRONG nhóm
        cnt[v] = cnt.get(v, 0) + 1   # 2) đếm số NHÓM
return [v for v in cnt if cnt[v] >= k]  # 3) lọc
```

Với đúng 3 tập: `(A & B) | (A & C) | (B & C)`.

**Checklist trước khi bấm Submit**

- [ ] Đã bỏ trùng **trong từng mảng** trước khi đếm chưa?
- [ ] Kết quả có phần tử trùng không? (dùng keys dict / set)
- [ ] Điều kiện là `>= 2` chứ không phải `== 3` hay `> 2`?
- [ ] Tên method đúng `twoOutOfThree`, trả về **list** (không phải set — LeetCode chấp nhận list; đổi bằng `list(...)` cho chắc).
- [ ] Đã thử ví dụ 3 (trùng trong cùng mảng → `[]`).